In [0]:
VOLUME_PATH   = "/Volumes/demo_projects/default/dynamic_regulatory_system"
CHUNK_TABLE   = "demo_projects.default.saib_chunks"
ENDPOINT      = "saib_vs"                      # reuse the original; delete saib_full (see note)
INDEX_NAME    = "demo_projects.default.saib_index"
EMBEDDER      = "databricks-gte-large-en"
LLM           = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
parsed_df = spark.sql(f"""
SELECT _metadata.file_name AS file_name,
       ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
FROM READ_FILES('{VOLUME_PATH}', format => 'binaryFile', fileNamePattern => '*.pdf')
""")

In [0]:
import re, json
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, StructType, StructField

def extract_fields(parsed_json_str):
    try:
        parsed = json.loads(parsed_json_str)
        elements = parsed.get('document', {}).get('elements', [])
        full_text = ''
        for e in elements:
            if e.get('type') in ['text','title','section_header'] and e.get('content'):
                full_text += e['content'] + '\n'

        def grab(pat, flags=re.IGNORECASE):
            m = re.search(pat, full_text, flags)
            return m.group(1).strip() if m else ''

        return {
            'saib_number': grab(r'SAIB[:\s]+([\w-]+)'),
            'subject':     grab(r'SUBJ[:\s]+([^\n]+)') or grab(r'Subject[:\s]+([^\n]+)'),
            'make':        grab(r'Make[:\s]+([^\n]+)'),
            'model':       grab(r'Model[s]?[:\s]+([^\n]+)'),
            'background':      grab(r'Background\s*\n+(.*?)(?=\n\s*(?:Recommendations|Discussion|For Further Information|For Related Service)\s*\n|$)', re.IGNORECASE|re.DOTALL),
            'recommendations': grab(r'Recommendations\s*\n+(.*?)(?=\n\s*(?:For Further Information|For Related Service|Figure)\s*\n|$)', re.IGNORECASE|re.DOTALL),
            'full_text': full_text.strip(),
        }
    except Exception as e:
        return {k: '' for k in ['saib_number','subject','make','model','background','recommendations']} | {'full_text': f'Error parsing: {e}'}

schema = StructType([StructField(c, StringType(), True) for c in
        ['saib_number','subject','make','model','background','recommendations','full_text']])
extract_udf = udf(extract_fields, schema)

extracted_df = (parsed_df
    .select('file_name', extract_udf(col('parsed_content').cast('string')).alias('x'))
    .select('file_name', 'x.*'))

In [0]:
from pyspark.sql import functions as F
extracted_df = extracted_df.withColumn(
    "saib_number",
    F.regexp_replace(F.col("file_name"), r"\.pdf$", "")
)

In [0]:
from pyspark.sql import functions as F

extracted_df = extracted_df.withColumn(
    "saib_number",
    F.regexp_replace(F.col("file_name"), r"\.pdf$", "")
).withColumn(
    "saib_number",
    F.regexp_replace(F.col("saib_number"), r"%20", " ")   # URL-decode space
).withColumn(
    "saib_number",
    F.regexp_replace(F.col("saib_number"), r"^DRS-", "")  # strip source prefix
)

extracted_df.select("file_name","saib_number").distinct().show(60, truncate=False)

+------------------+------------+
|file_name         |saib_number |
+------------------+------------+
|2022-20.pdf       |2022-20     |
|DRS-2026-13.pdf   |2026-13     |
|AIR-21-08.pdf     |AIR-21-08   |
|AIR-21-10.pdf     |AIR-21-10   |
|DRS-2026-16.pdf   |2026-16     |
|AIR-22-15.pdf     |AIR-22-15   |
|DRS-2026-12.pdf   |2026-12     |
|AIR-22-04.pdf     |AIR-22-04   |
|DRS-2026-14.pdf   |2026-14     |
|AIR-22-10.pdf     |AIR-22-10   |
|2026-10R1.pdf     |2026-10R1   |
|DRS-2026-11.pdf   |2026-11     |
|AIR-22-09R1.pdf   |AIR-22-09R1 |
|AIR-21-19R1.pdf   |AIR-21-19R1 |
|CE-19-01%20R1.pdf |CE-19-01 R1 |
|2026-05.pdf       |2026-05     |
|2026-06.pdf       |2026-06     |
|NE-06-31R6.pdf    |NE-06-31R6  |
|DRS-2026-15.pdf   |2026-15     |
|AIR-22-03.pdf     |AIR-22-03   |
|2022-16.pdf       |2022-16     |
|AIR-21-20.pdf     |AIR-21-20   |
|CE-13-27R1.pdf    |CE-13-27R1  |
|2025-09.pdf       |2025-09     |
|2026-07.pdf       |2026-07     |
|2025-11.pdf       |2025-11     |
|AIR-22-08.pdf

In [0]:
import pyspark.sql.functions as F
total = extracted_df.count()
failed = extracted_df.filter(col('full_text').startswith('Error parsing')).count()
empty  = extracted_df.filter(col('full_text').isNull() | (F.length('full_text') < 100)).count()
print(f"total={total}  failed={failed}  empty={empty}  usable={total-failed-empty}")
extracted_df.filter(col('full_text').startswith('Error parsing')).select('file_name').show(truncate=False)

total=56  failed=0  empty=0  usable=56
+---------+
|file_name|
+---------+
+---------+



In [0]:
extracted_df.display()

file_name saib_number subject make model background recommendations full_text DRS-2026-16.pdf 2026-16 Elevator Control System Date: September 10, 2026 99, 99A, 99A (FACH), A99, A99A, B99, C99, 100, A100, and B100 airplanes of the potential for corrosion in the elevator torque shafts. Some of these models are commercially referred to as King Air and Airliner models. The FAA received a report of extensive corrosion on the inside of an elevator torque shaft on a Textron Aviation Inc. King Air Model A100 airplane that was manufactured in 1977. During a scheduled inspection, the elevator torque shaft was found to show signs of surface corrosion. Further investigation revealed extensive corrosion on the inside of the elevator torque shaft as shown in Figure 1. It was determined that the rig pin hole on the elevator torque shaft assembly is susceptible to water ingestion and accumulation. This condition has the potential to cause corrosion on the inner diameter of the elevator torque shaft and on the taper pins, as shown in Figure 2. A contributing factor to the severity of the corrosion is the absence of an established inspection procedure for the interior of the elevator torque shaft. The FAA recommends inspection and replacement of the elevator torque shaft assembly in accordance with Beechcraft Mandatory Service Letter MTL-27-03, dated June 6, 2023.

In addition, the FAA recommends reporting any findings of cracked, corroded, or damaged elevator torque shaft assemblies found during the inspection via email to ccb-cos@faa.gov. Please include the make, model, serial number, and total time in service of the aircraft. If possible, attach photographs of cracked, corroded, or damaged locations.

Under the provisions of the Paperwork Reduction Act (44 U.S.C. 3501 et seq.), the OMB has approved the information collection contained in this SAIB and assigned OMB Control Number 2120-0731. All responses to this collection of information are voluntary; the nature and extent of confidentiality to be provided, if any. Send comments regarding this burden estimate or any other aspect of this collection of information, including suggestions for reducing this burden to:
Information Collection Clearance Officer, Federal Aviation Administration, 10101 Hillwood Parkway, Fort Worth, TX 76177-1524.
For Further Information Contact
Soban Saeed, Aviation Safety Engineer, 1801 S Airport Road, Wichita, KS 67209; phone: (316) 946-4123; email: ccb-cos@faa.gov.
For Related Service Information Contact
Textron Aviation, 2121 S Hoover Road, Wichita, KS 67209; phone: (316) 517-1700; email: teamturboprop@txtav.com.
Corrosion on the inside of elevator torque shaft
Corroded taper pins FAA
Aviation Safety
SPECIAL AIRWORTHINESS
INFORMATION BULLETIN
SAIB: 2026-16
SUBJ: Elevator Control System Date: September 10, 2026
This is information only. Recommendations aren't mandatory.
Introduction
This Special Airworthiness Information Bulletin advises owners, operators, and maintenance technicians of Textron Aviation Inc. (type certificate formerly held by Beechcraft Corporation, Hawker Beechcraft, Raytheon Aircraft, and Beech Aircraft Corporation) Model 99, 99A, 99A (FACH), A99, A99A, B99, C99, 100, A100, and B100 airplanes of the potential for corrosion in the elevator torque shafts. Some of these models are commercially referred to as King Air and Airliner models.

At this time, the airworthiness concern is not an unsafe condition that would warrant airworthiness directive (AD) action under Title 14 of the Code of Federal Regulations (14 CFR) part 39.
Background
The FAA received a report of extensive corrosion on the inside of an elevator torque shaft on a Textron Aviation Inc. King Air Model A100 airplane that was manufactured in 1977. During a scheduled inspection, the elevator torque shaft was found to show signs of surface corrosion. Further investigation revealed extensive corrosion on the inside of the elevator torque shaft as shown in Figure 1. It was determined that t

In [0]:
from pyspark.sql import functions as F
chunks = (extracted_df
    .filter(~col('full_text').startswith('Error parsing'))     # drop failed rows
    .select('saib_number', 'subject', F.col('full_text').alias('chunk'))
    .withColumn('id', F.monotonically_increasing_id()))

(chunks.write.format('delta')
    .option('delta.enableChangeDataFeed', 'true')
    .mode('overwrite').option('overwriteSchema','true')
    .saveAsTable(CHUNK_TABLE))

print("chunk table rows:", spark.table(CHUNK_TABLE).count())   # expect ~56 minus failures

chunk table rows: 56


In [0]:
%pip install databricks-vectorsearch

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import time
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

# create the endpoint (fresh Azure workspace — nothing exists yet)
existing = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if ENDPOINT not in existing:
    vsc.create_endpoint(name=ENDPOINT, endpoint_type="STANDARD")

# wait until ONLINE before creating the index
while True:
    s = vsc.get_endpoint(ENDPOINT)["endpoint_status"]["state"]
    print("endpoint:", s)
    if s == "ONLINE":
        break
    time.sleep(30)
print("endpoint ready")

/home/spark-94dcb211-6b71-4857-a986-5f/.ipykernel/78/command-6455547200297287-1946436645:2: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


endpoint: ONLINE
endpoint ready


In [0]:
import time
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

# delete the old index if it exists (Free Edition = one unit), wait until gone
try:
    vsc.delete_index(index_name=INDEX_NAME)
    while True:
        try:
            vsc.get_index(index_name=INDEX_NAME).describe(); time.sleep(10)
        except Exception:
            break   # gone
    print("old index deleted")
except Exception as e:
    print("no old index / already gone:", e)

index = vsc.create_delta_sync_index(
    endpoint_name=ENDPOINT,
    index_name=INDEX_NAME,
    source_table_name=CHUNK_TABLE,
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_source_column="chunk",
    embedding_model_endpoint_name=EMBEDDER)

no old index / already gone: Unity Catalog entity demo_projects.default.saib_index does not exist.


In [0]:
print(CHUNK_TABLE, "|", INDEX_NAME)

demo_projects.default.saib_chunks | demo_projects.default.saib_index


In [0]:
for s in ["AIR-21-07", "AIR-21-09", "2026-15"]:
    r = spark.table("demo_projects.default.saibs_raw").filter(f"saib_number = '{s}'") \
          .select("subject","background","recommendations").first()
    print(f"\n===== {s} — {r['subject']} =====")
    print("BG:", r["background"][:300])
    print("REC:", r["recommendations"][:200])


===== AIR-21-07 — WING SPAR, Verification of Correct Washers used with Wing Spar =====
BG: AD 2020-26-16 requires the replacement of four wing spar attach bolts following eddy current
inspection of the subject bolt holes. Although not required by the AD, it is common practice to
install new washers with the new bolts. Some of the washers listed in the various model Illustrated
Parts Catal
REC: The FAA recommends that any owner/operator ordering and receiving Piper part number 96352-003
(Piper purchase order code 494-284) washers for use in the wing spar attach bolts of any aircraft
applicab

===== AIR-21-09 — LANDING GEAR SYSTEM, Landing gear skid tube =====
BG: A specific French operator provides EC 120 B helicopters to military pilots for training purposes.
The training exercises include using the aircraft for a high number of autorotation landings with full
touchdown completed. Cracks were found on a landing gear skid tube of one of the French operator’s
REC: The Federal Aviation A

In [0]:
import pandas as pd
eval_set = [
    ("cracks on EC120B helicopter landing gear skid tubes", "AIR-21-09", "clean"),
    ("incorrect flat washers on Piper wing spar attach bolts", "AIR-21-07", "clean"),
    ("rupture disc on evacuation slide activating below 160F", "AIR-22-06R1", "clean"),
    ("autopilot randomly swings the nose to north during approach", "AIR-22-07", "vague"),
    ("my plane stopped showing up on ATC radar after I powered on avionics", "AIR-21-15 R1", "vague"),
    ("helicopter skids cracking after lots of practice landings", "AIR-21-09", "vague"),
    ("washers I got don't have the rounded edge the manual shows", "AIR-21-07", "vague"),
    ("GPS navigation problem causing wrong heading", "AIR-22-07", "cross-doc"),
    ("cracking found on a structural airframe part", "2026-15", "cross-doc"),
    ("ADS-B or transponder not transmitting", "AIR-21-15 R1", "cross-doc"),
    ("what is the correct tyre pressure for a Cessna 172", "NOT_FOUND", "no-answer"),
    ("how often should I change engine oil on a Boeing 737", "NOT_FOUND", "no-answer"),
    ("what is the maximum takeoff weight of an Airbus A320", "NOT_FOUND", "no-answer"),
]
df = pd.DataFrame(eval_set, columns=["query","expected","category"])
df["also_valid"] = ""
spark.createDataFrame(df).write.format("delta").mode("overwrite") \
    .saveAsTable("demo_projects.default.saib_eval_set")
print(spark.table("demo_projects.default.saib_eval_set").count(), "rows")

13 rows


In [0]:
from pyspark.sql import functions as F
t = spark.table("demo_projects.default.saib_eval_set")
t = t.withColumn("also_valid",
      F.when(F.col("query")=="cracking found on a structural airframe part", F.lit("AIR-21-09"))
       .otherwise(F.col("also_valid")))
t.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable("demo_projects.default.saib_eval_set")
t.filter("category='cross-doc'").show(truncate=False)

+--------------------------------------------+------------+---------+----------+
|query                                       |expected    |category |also_valid|
+--------------------------------------------+------------+---------+----------+
|GPS navigation problem causing wrong heading|AIR-22-07   |cross-doc|          |
|cracking found on a structural airframe part|2026-15     |cross-doc|AIR-21-09 |
|ADS-B or transponder not transmitting       |AIR-21-15 R1|cross-doc|          |
+--------------------------------------------+------------+---------+----------+



In [0]:
import pandas as pd
df = pd.DataFrame(eval_set, columns=["query","expected","category"])
df["also_valid"] = ""

m = df["query"] == "cracking found on a structural airframe part"
df.loc[m, "expected"]   = "2026-15"
df.loc[m, "also_valid"] = "AIR-21-09"

spark.createDataFrame(df).write.format("delta").mode("overwrite") \
    .saveAsTable("demo_projects.default.saib_eval_set")
print(spark.table("demo_projects.default.saib_eval_set").count(), "rows written")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7902060935100914>, line 2
      1 import pandas as pd
----> 2 df = pd.DataFrame(eval_set, columns=["query","expected","category"])
      3 df["also_valid"] = ""
      5 m = df["query"] == "cracking found on a structural airframe part"

NameError: name 'eval_set' is not defined

In [0]:
import time
while not index.describe()["status"]["ready"]:
    print("indexing…", index.describe()["status"].get("indexed_row_count",0)); time.sleep(30)
print("READY — indexed:", index.describe()["status"]["indexed_row_count"],
      "| chunk table:", spark.table(CHUNK_TABLE).count())

indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0


indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0
indexing… 0


In [0]:
# rewrite saib_chunks (index source), then:
import time
from databricks.ai_search.utils import BadRequest
index = vsc.get_index(index_name=INDEX_NAME)

# wait until the index is ready before syncing
while not index.describe()["status"].get("ready", False):
    s = index.describe()["status"]
    print(f"waiting… indexed={s.get('indexed_row_count', 0)}")
    time.sleep(30)

# sync with retry in case a pipeline is still finishing
for attempt in range(20):
    try:
        index.sync()
        print("sync triggered")
        break
    except BadRequest as e:
        if "not ready to sync yet" in str(e).lower() or "pipeline is in state running" in str(e).lower():
            print(f"pipeline still running, retrying in 30s… (attempt {attempt+1})")
            time.sleep(30)
        else:
            raise

sync triggered


In [0]:
import time
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)
if ENDPOINT not in [e["name"] for e in vsc.list_endpoints().get("endpoints",[])]:
    vsc.create_endpoint(name=ENDPOINT, endpoint_type="STANDARD")
print("provisioning started — don't wait, do Step 1")

provisioning started — don't wait, do Step 1


In [0]:
print("endpoint:", vsc.get_endpoint(ENDPOINT)["endpoint_status"]["state"])
print("index:", index.describe()["status"])

endpoint: ONLINE
index: {'detailed_state': 'ONLINE_NO_PENDING_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://adb-7405614992177253.13.azuredatabricks.net/explore/data/demo_projects/default/saib_index', 'indexed_row_count': 56, 'triggered_update_status': {'last_processed_commit_version': 1, 'last_processed_commit_timestamp': '2026-10-13T05:47:44Z'}, 'ready': True, 'index_url': 'adb-7405614992177253.13.azuredatabricks.net/api/2.0/vector-search/indexes/demo_projects.default.saib_index'}


In [0]:
from pyspark.sql.functions import col
(extracted_df.filter(~col('full_text').startswith('Error parsing'))
   .write.format('delta').mode('overwrite').option('overwriteSchema','true')
   .saveAsTable('demo_projects.default.saibs_raw'))
print("saibs_raw:", spark.table("demo_projects.default.saibs_raw").count())


saibs_raw: 56


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
w = WorkspaceClient()

SYSTEM = ("You answer questions using ONLY the SAIB documents in the context. "
          "Do not use outside knowledge. Each document is labeled with its SAIB number. "
          "Cite the exact SAIB number(s) you used as 'SAIB <number>'. "
          "If the answer is not in the documents, reply with exactly 'NOT_FOUND' and nothing else.")

def rag_answer(query, top_k=3):
    rows = index.similarity_search(
        query_text=query,
        columns=["saib_number", "chunk"],
        num_results=top_k)["result"]["data_array"]
    sources = [r[0] for r in rows]
    context = "\n\n---\n\n".join(f"SAIB {r[0]}:\n{r[1]}" for r in rows)
    resp = w.serving_endpoints.query(
        name=LLM, temperature=0.0,
        messages=[ChatMessage(role=ChatMessageRole.SYSTEM, content=SYSTEM),
                  ChatMessage(role=ChatMessageRole.USER,
                              content=f"Context:\n{context}\n\nQuestion: {query}")])
    return {"answer": resp.choices[0].message.content, "sources": sources}

In [0]:
queries = [
    "incorrect flat washers on wing spar attach bolts",          # AIR-21-07
    "cracks on helicopter landing gear skid tubes",              # AIR-21-09
    "ADS-B Out stops transmitting after avionics power up",      # AIR-21-15
    "autopilot uncommanded turn to heading 360 on LPV approach", # AIR-22-07
    "rupture disc on evacuation slide activating when too cold", # AIR-22-06
]
for q in queries:
    out = rag_answer(q)
    print("\nQ:", q)
    print("SOURCES:", out["sources"])
    print("ANSWER:", out["answer"][:160])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

Q: incorrect flat washers on wing spar attach bolts
SOURCES: ['AIR-21-07', 'AIR-21-16', 'AIR-21-10']
ANSWER: The issue of incorrect flat washers on wing spar attach bolts is addressed in SAIB AIR-21-07. According to this document, Piper Aircraft, Inc. has made the FAA 
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

Q: cracks on helicopter landing gear skid tubes
SOURCES: ['AIR-21-09', '', '2026-14']
ANSWER: Cracks have been found on left-hand (LH) and right-hand (RH) landing gear skid tubes of Airbus Helicopters (formerly Eurocopter France) Model EC 120 B helicopte
[NOTICE] Using a notebook authentication token. Recom

In [0]:
spark.table(CHUNK_TABLE).filter("saib_number = '' OR saib_number IS NULL").count()

1

In [0]:
spark.table("demo_projects.default.saibs_raw") \
    .filter("saib_number = '' OR saib_number IS NULL") \
    .select("file_name","full_text").show(truncate=120)

+-------------+------------------------------------------------------------------------------------------------------------------------+
|    file_name|                                                                                                               full_text|
+-------------+------------------------------------------------------------------------------------------------------------------------+
|AIR-22-08.pdf|FAA\nAviation Safety\nSPECIAL AIRWORTHINESS\nINFORMATION BULLETIN\nSUBJ: MAIN ROTOR DRIVE SYSTEM,TAIL ROTOR DRIVE SYS...|
+-------------+------------------------------------------------------------------------------------------------------------------------+



In [0]:
extracted_df.select("file_name","saib_number").distinct().show(60, truncate=False)

+------------------+--------------+
|file_name         |saib_number   |
+------------------+--------------+
|2022-20.pdf       |2022-20       |
|DRS-2026-16.pdf   |DRS-2026-16   |
|AIR-21-08.pdf     |AIR-21-08     |
|DRS-2026-13.pdf   |DRS-2026-13   |
|AIR-21-10.pdf     |AIR-21-10     |
|AIR-22-15.pdf     |AIR-22-15     |
|DRS-2026-14.pdf   |DRS-2026-14   |
|AIR-22-04.pdf     |AIR-22-04     |
|AIR-22-10.pdf     |AIR-22-10     |
|DRS-2026-12.pdf   |DRS-2026-12   |
|DRS-2026-11.pdf   |DRS-2026-11   |
|2026-10R1.pdf     |2026-10R1     |
|AIR-22-09R1.pdf   |AIR-22-09R1   |
|DRS-2026-15.pdf   |DRS-2026-15   |
|AIR-21-19R1.pdf   |AIR-21-19R1   |
|CE-19-01%20R1.pdf |CE-19-01%20R1 |
|2026-05.pdf       |2026-05       |
|2026-06.pdf       |2026-06       |
|NE-06-31R6.pdf    |NE-06-31R6    |
|AIR-22-03.pdf     |AIR-22-03     |
|2022-16.pdf       |2022-16       |
|AIR-21-20.pdf     |AIR-21-20     |
|CE-13-27R1.pdf    |CE-13-27R1    |
|2025-09.pdf       |2025-09       |
|2026-07.pdf       |2026-07 

In [0]:
# === SESSION CLOSE 2026-10-13 ===
# S5 DONE. Corpus 56 docs, Azure demo_projects.default.
# Fixed overnight-stuck index (delete+recreate against ONLINE endpoint).
# Integrity gate PASSED: index=56, chunk=56, saibs_raw=56.
# 5 eyeball queries: ALL correct doc at #1, grounded + cited. Retrieval works at scale.
# S6 SEED (findings): (1) one chunk has blank saib_number — extraction miss, fix it.
#   (2) top_k=3 feeds 2 noise chunks/query; all test queries were easy keyword matches —
#       S6 needs adversarial queries (vague/cross-doc/no-answer) to test precision.
# ENDPOINT DELETED — ~Rs800/day idle. Tables + index def persist in UC.
# RESUME: recreate endpoint → wait ONLINE → get_index → query. (Create index ONLY after endpoint ONLINE.)
vsc.delete_endpoint(name=ENDPOINT)
print("endpoint deleted:", ENDPOINT) 

endpoint deleted: saib_vs


In [0]:
import time
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

index = vsc.get_index(index_name=INDEX_NAME)
print(index.describe()["status"]["ready"], "|", index.describe()["status"]["indexed_row_count"])

/home/spark-88c9a7a8-50c3-404e-9af0-9e/.ipykernel/78/command-5920460044979343-3997357859:2: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


True | 56


In [0]:
import pandas as pd
evdf = spark.table("demo_projects.default.saib_eval_set").toPandas()

rows = []
for _, r in evdf.iterrows():
    out = rag_answer(r["query"], top_k=3)
    expected, also = r["expected"], (r["also_valid"] or "")
    valid = {x for x in [expected, also] if x}
    ans = out["answer"].strip()

    hit = (len(valid & set(out["sources"])) > 0) if r["category"] != "no-answer" else None
    if r["category"] == "no-answer":
        correct = ans == "NOT_FOUND"
    else:
        correct = any(v in ans for v in valid) and ans != "NOT_FOUND"

    rows.append({"query": r["query"], "category": r["category"], "expected": expected,
                 "sources": out["sources"], "hit": hit, "correct": correct, "answer": ans[:80]})

res = pd.DataFrame(rows)

for cat in ["clean","vague","cross-doc","no-answer"]:
    c = res[res["category"]==cat]
    if cat == "no-answer":
        print(f"{cat:10} refusal: {c['correct'].sum()}/{len(c)}")
    else:
        print(f"{cat:10} retrieval-hit: {c['hit'].sum()}/{len(c)}  answer-correct: {c['correct'].sum()}/{len(c)}")

print("\n--- weak (document, don't fix today) ---")
print(res[(res['hit']==False) | (res['correct']==False)][["query","category","expected","sources","answer"]].to_string())

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

In [0]:
spark.table("demo_projects.default.saib_chunks").select("saib_number").distinct().show(60, truncate=False)

+--------------+
|saib_number   |
+--------------+
|2022-16       |
|2022-20       |
|2023-03R2     |
|2022-21       |
|2022-23       |
|2025-10       |
|2025-07       |
|2025-06       |
|2026-05       |
|2026-06       |
|2025-09       |
|2026-08       |
|2025-11       |
|2026-09       |
|2026-04       |
|2025-08       |
|2026-02       |
|AIR-21-20     |
|2026-07       |
|NE-06-31R6    |
|AIR-21-09     |
|AIR-21-05R1   |
|AIR-22-04     |
|AIR-21-12     |
|AIR-22-03     |
|AIR-21-14     |
|AIR-22-02R1   |
|AIR-21-07     |
|AIR-21-15%20R1|
|AIR-21-03     |
|AIR-21-13     |
|AIR-22-07     |
|AIR-22-08     |
|CE-13-27R1    |
|DRS-2026-15   |
|DRS-2026-14   |
|DRS-2026-11   |
|DRS-2026-13   |
|AIR-21-19R1   |
|AIR-21-10     |
|2026-10R1     |
|NM-18-32%20R1 |
|CE-19-01%20R1 |
|AIR-22-09R1   |
|AIR-22-10     |
|AIR-21-08     |
|AIR-22-13     |
|AIR-22-15     |
|SW-17-31R4    |
|DRS-2026-16   |
|AIR-21-16     |
|AIR-21-04     |
|2026-01       |
|DRS-2026-12   |
|AIR-22-06R1   |
|HQ-14-16R1   

In [0]:
from pyspark.sql import functions as F

fixed = (spark.table("demo_projects.default.saib_chunks")
    .withColumn("saib_number", F.regexp_replace("saib_number", r"%20", " "))
    .withColumn("saib_number", F.regexp_replace("saib_number", r"^DRS-", "")))

fixed.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable("demo_projects.default.saib_chunks")

index = vsc.get_index(index_name=INDEX_NAME)
index.sync()
print("syncing — wait for ready before re-running eval")

syncing — wait for ready before re-running eval


In [0]:
import time
while not index.describe()["status"]["ready"]:
    time.sleep(20)
print("ready:", index.describe()["status"]["indexed_row_count"])  # 56

ready: 56


In [0]:
rows = []
for _, r in evdf.iterrows():
    out = rag_answer(r["query"], top_k=3)
    valid = {x for x in [r["expected"], (r["also_valid"] or "")] if x}
    ans = out["answer"].strip()
    hit = (len(valid & set(out["sources"])) > 0) if r["category"]!="no-answer" else None
    correct = (ans=="NOT_FOUND") if r["category"]=="no-answer" else (any(v in ans for v in valid) and ans!="NOT_FOUND")
    rows.append({"query":r["query"],"category":r["category"],"sources":out["sources"],"hit":hit,"correct":correct})
res = pd.DataFrame(rows)

for cat in ["clean","vague","cross-doc","no-answer"]:
    c = res[res["category"]==cat]
    if cat=="no-answer": print(f"{cat:10} refusal: {c['correct'].sum()}/{len(c)}")
    else: print(f"{cat:10} hit: {c['hit'].sum()}/{len(c)}  correct: {c['correct'].sum()}/{len(c)}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T